# Módulo 40: SQLite con Python

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio de este módulo: [enunciado](m40-sqlite-con-python-guia.ipynb) y [solución](m40-sqlite-con-python-solucion.ipynb). No es una tarea entregada del programa.

En los módulos anteriores escribimos SQL para SQL Server. Ahora lo usamos **desde Python**, con el módulo `sqlite3` de la librería estándar. Así un mismo programa puede leer un CSV, guardarlo en una base, consultarla y llevar el resultado a pandas. Lo importante no es solo que funcione, sino hacerlo de forma segura: con parámetros, transacciones y conexiones bien cerradas.

1. Por qué SQLite
2. El flujo básico: conexión, cursor, ejecución
3. Transacciones: `commit`, `rollback` y `with`
4. Parámetros `?` frente a f-strings
5. `executemany`
6. Cargar un CSV
7. Leer resultados
8. Respaldar con `iterdump`
9. Errores frecuentes
10. Ejercicio

## 1. Por qué SQLite

SQLite es un motor de base de datos que **no necesita servidor**: toda la base vive en un único archivo (o en memoria) y la librería se ejecuta dentro del propio programa.

| Característica | SQLite | SQL Server |
| --- | --- | --- |
| Instalación | Ninguna: viene con Python | Servidor que se instala y administra |
| Almacenamiento | Un archivo `.db` | Archivos gestionados por el servidor |
| Usuarios simultáneos | Muchos lectores, un escritor a la vez | Muchos lectores y escritores |
| Usos típicos | Prototipos, análisis local, apps móviles, pruebas | Sistemas de empresa con muchos usuarios |

Para análisis de datos en una sola máquina es ideal: se crea con una línea y se comparte copiando un archivo. En esta lección usamos `":memory:"`, una base que vive en la RAM y desaparece al cerrar la conexión.

## 2. El flujo básico: conexión, cursor, ejecución

| Paso | Código | Para qué |
| --- | --- | --- |
| 1 | `conn = sqlite3.connect(ruta)` | Abrir (o crear) la base |
| 2 | `cur = conn.cursor()` | Obtener un cursor, que ejecuta sentencias y recorre resultados |
| 3 | `cur.execute(sql, parámetros)` | Ejecutar una sentencia |
| 4 | `cur.fetchone()` / `cur.fetchall()` | Leer filas de un `SELECT` |
| 5 | `conn.commit()` | Confirmar los cambios |
| 6 | `conn.close()` | Cerrar la conexión |

`conn.execute(...)` es un atajo que crea el cursor por nosotros y lo devuelve.

In [1]:
import csv
import io
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path

import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.execute("""
CREATE TABLE Producto (
    ProductoID INTEGER PRIMARY KEY,
    Nombre     TEXT NOT NULL,
    Precio     REAL
)""")
cur.execute("INSERT INTO Producto (Nombre, Precio) VALUES ('Casco', 34.99)")
cur.execute("INSERT INTO Producto (Nombre, Precio) VALUES ('Gorra', 8.99)")
conn.commit()

cur.execute("SELECT ProductoID, Nombre, Precio FROM Producto")
print(cur.fetchall())   # lista de tuplas
print("Último id generado:", cur.lastrowid)

[(1, 'Casco', 34.99), (2, 'Gorra', 8.99)]
Último id generado: 2


Una columna `INTEGER PRIMARY KEY` se numera sola si no se le da valor, y `cursor.lastrowid` devuelve el último número asignado.

## 3. Transacciones: `commit`, `rollback` y `with`

Los cambios (`INSERT`, `UPDATE`, `DELETE`) quedan **pendientes** hasta llamar a `commit()`. Con `rollback()` se descartan. Esto permite agrupar varios cambios que deben aplicarse todos o ninguno.

Usar la conexión en un bloque `with` automatiza esa decisión: si el bloque termina bien, hace `commit`; si se lanza una excepción, hace `rollback`. **No cierra la conexión**; para eso se combina con `contextlib.closing` o se llama a `close()`.

In [2]:
try:
    with conn:   # commit si todo sale bien, rollback si hay excepción
        conn.execute("INSERT INTO Producto (Nombre, Precio) VALUES ('Calcetines', 9.50)")
        conn.execute("INSERT INTO Producto (Nombre, Precio) VALUES (NULL, 5.00)")   # viola NOT NULL
except sqlite3.IntegrityError as error:
    print("Se deshizo todo el bloque:", error)

print(conn.execute("SELECT COUNT(*) FROM Producto").fetchone()[0], "productos")

Se deshizo todo el bloque: NOT NULL constraint failed: Producto.Nombre
2 productos


Los calcetines tampoco se guardaron: el bloque entero se revirtió porque la segunda inserción falló.

## 4. Parámetros `?` frente a f-strings

Cuando un valor viene de afuera (un formulario, un archivo, otra función), **no se pega dentro del texto SQL**. Se escribe un marcador `?` y el valor se pasa aparte, en una tupla. El motor lo trata siempre como un dato, nunca como código.

Con un f-string, el valor pasa a formar parte de la sentencia. Si contiene comillas, la sentencia se rompe; y si alguien lo prepara con mala intención, puede cambiar la consulta. Eso se llama **inyección SQL**.

In [3]:
conn.executescript("""
CREATE TABLE Usuario (Login TEXT PRIMARY KEY, Rol TEXT);
INSERT INTO Usuario VALUES ('ana', 'admin'), ('luis', 'lector'), ('marta', 'lector');
""")

entrada = "nadie' OR '1'='1"     # texto que escribe un usuario malicioso

# Mal: el texto se pega en la sentencia y la condición pasa a ser siempre verdadera
sql_inseguro = f"SELECT Login, Rol FROM Usuario WHERE Login = '{entrada}'"
print(sql_inseguro)
print("f-string:", conn.execute(sql_inseguro).fetchall())

# Bien: el texto completo se compara como un valor
print("parámetro:", conn.execute("SELECT Login, Rol FROM Usuario WHERE Login = ?", (entrada,)).fetchall())

SELECT Login, Rol FROM Usuario WHERE Login = 'nadie' OR '1'='1'
f-string: [('ana', 'admin'), ('luis', 'lector'), ('marta', 'lector')]
parámetro: []


Además de `?`, `sqlite3` acepta **parámetros con nombre** (`:login`) y un diccionario. Son útiles cuando hay muchos valores:

In [4]:
conn.execute("SELECT Login, Rol FROM Usuario WHERE Rol = :rol AND Login <> :excluir",
             {"rol": "lector", "excluir": "luis"}).fetchall()

[('marta', 'lector')]

Los parámetros sirven para **valores**. Los nombres de tablas o columnas no se pueden parametrizar; si dependen de una variable, hay que validarlos contra una lista de nombres permitidos antes de usarlos.

## 5. `executemany`

`executemany(sql, secuencia)` ejecuta la misma sentencia una vez por cada elemento de la secuencia. Es más rápido y más claro que un bucle con `execute`, y funciona con tuplas (para `?`) o con diccionarios (para `:nombre`).

In [5]:
nuevos = [("Jersey", 49.99), ("Botella", 4.99), ("Guantes", 24.49)]
with conn:
    conn.executemany("INSERT INTO Producto (Nombre, Precio) VALUES (?, ?)", nuevos)

pd.read_sql_query("SELECT * FROM Producto", conn)

,ProductoID,Nombre,Precio
0,1,Casco,34.99
1,2,Gorra,8.99
2,3,Jersey,49.99
3,4,Botella,4.99
4,5,Guantes,24.49


## 6. Cargar un CSV

El archivo `customers.csv` del módulo tiene 91 clientes. Hay dos formas habituales de cargarlo:

- Con el módulo `csv` y `executemany`: control total sobre la tabla y los tipos.
- Con pandas (`read_csv` + `to_sql`), que se verá en el Módulo 41.

`csv.DictReader` produce un diccionario por fila, con los encabezados como claves, así que encaja directamente con parámetros con nombre. Los valores llegan como texto; la afinidad de las columnas `INTEGER` los convierte a número.

In [6]:
conn.execute("""
CREATE TABLE Cliente (
    CustomerID   INTEGER PRIMARY KEY,
    CustomerName TEXT,
    ContactName  TEXT,
    Address      TEXT,
    City         TEXT,
    PostalCode   TEXT,
    Country      TEXT,
    PurchaseUSD  INTEGER
)""")

with open("customers.csv", encoding="utf-8", newline="") as archivo:
    lector = csv.DictReader(archivo)
    with conn:
        conn.executemany("""
            INSERT INTO Cliente VALUES (:CustomerID, :CustomerName, :ContactName, :Address,
                                        :City, :PostalCode, :Country, :PurchaseUSD)""", lector)

print(conn.execute("SELECT COUNT(*) FROM Cliente").fetchone()[0], "clientes cargados")
print(conn.execute("SELECT typeof(CustomerID), typeof(PurchaseUSD), typeof(PostalCode) FROM Cliente LIMIT 1").fetchone())

91 clientes cargados
('integer', 'integer', 'text')


`PostalCode` se declara como `TEXT` a propósito: algunos códigos postales tienen letras (`WA1 1DP`) o ceros a la izquierda que se perderían como número.

En el archivo, algunas letras acentuadas aparecen como `?` (por ejemplo, `Taquer?a`). El daño viene del archivo original, no de SQLite: el texto se guardó con una codificación que no tenía esos caracteres.

## 7. Leer resultados

| Método | Devuelve |
| --- | --- |
| `cur.fetchone()` | La siguiente fila, o `None` si no quedan |
| `cur.fetchall()` | Todas las filas restantes, en una lista |
| `for fila in cur:` | Las filas una a una, sin cargarlas todas en memoria |
| `pd.read_sql_query(sql, conn, params=...)` | Un DataFrame |

Por defecto cada fila es una tupla. Con `conn.row_factory = sqlite3.Row`, las filas permiten acceder a las columnas por nombre.

In [7]:
cur = conn.execute("SELECT COUNT(*), ROUND(AVG(PurchaseUSD), 2) FROM Cliente WHERE Country = ?", ("Mexico",))
cantidad, promedio = cur.fetchone()
print(f"México: {cantidad} clientes, compra promedio {promedio} USD")

conn.row_factory = sqlite3.Row
fila = conn.execute("SELECT CustomerName, City FROM Cliente WHERE CustomerID = ?", (11,)).fetchone()
print(fila["CustomerName"], "-", fila["City"])
conn.row_factory = None   # volver a tuplas

México: 5 clientes, compra promedio 10444.2 USD
B's Beverages - London


In [8]:
pd.read_sql_query("""
SELECT Country, COUNT(*) AS clientes, SUM(PurchaseUSD) AS compras
FROM Cliente
GROUP BY Country
HAVING COUNT(*) >= ?
ORDER BY compras DESC
""", conn, params=(7,))

,Country,clientes,compras
0,USA,13,175894
1,Germany,11,134598
2,France,11,134001
3,UK,7,105992
4,Brazil,9,81047


## 8. Respaldar con `iterdump`

`conn.iterdump()` genera, línea a línea, las sentencias SQL (`CREATE TABLE`, `INSERT`...) que reconstruyen la base completa. Guardadas en un archivo `.sql`, sirven de respaldo legible y portable. Aquí las escribimos en un `io.StringIO` para no crear archivos, y luego restauramos el respaldo en una base nueva.

In [9]:
respaldo = io.StringIO()
for linea in conn.iterdump():
    respaldo.write(linea + "\n")

texto = respaldo.getvalue()
print(texto[:300])

copia = sqlite3.connect(":memory:")
copia.executescript(texto)
print("Clientes en la copia:", copia.execute("SELECT COUNT(*) FROM Cliente").fetchone()[0])
copia.close()

BEGIN TRANSACTION;
CREATE TABLE Cliente (
    CustomerID   INTEGER PRIMARY KEY,
    CustomerName TEXT,
    ContactName  TEXT,
    Address      TEXT,
    City         TEXT,
    PostalCode   TEXT,
    Country      TEXT,
    PurchaseUSD  INTEGER
);
INSERT INTO "Cliente" VALUES(1,'Alfreds Futterkiste','
Clientes en la copia: 91


Para copiar una base a otra sin pasar por texto existe también `conn.backup(destino)`, que es más rápido en bases grandes.

## 9. Errores frecuentes

**Construir la consulta con un f-string.** Además del riesgo de inyección, cualquier valor con comillas rompe la sentencia. El cliente 11 se llama `B's Beverages`:

In [10]:
nombre = "B's Beverages"
try:
    conn.execute(f"SELECT City FROM Cliente WHERE CustomerName = '{nombre}'")
except sqlite3.OperationalError as error:
    print("f-string:", error)

print("parámetro:", conn.execute("SELECT City FROM Cliente WHERE CustomerName = ?", (nombre,)).fetchone())

f-string: near "s": syntax error
parámetro: ('London',)


**Pasar un solo parámetro sin tupla.** `("lector")` es un texto, no una tupla: `sqlite3` lo recorre letra por letra y cree que hay seis parámetros. La tupla de un elemento lleva coma: `("lector",)`.

In [11]:
try:
    conn.execute("SELECT Login FROM Usuario WHERE Rol = ?", ("lector"))
except sqlite3.ProgrammingError as error:
    print("Sin coma:", error)

conn.execute("SELECT Login FROM Usuario WHERE Rol = ?", ("lector",)).fetchall()

Sin coma: Incorrect number of bindings supplied. The current statement uses 1, and there are 6 supplied.


[('luis',), ('marta',)]

**Olvidar el `commit`.** Al cerrar la conexión, los cambios pendientes se pierden. Para mostrarlo hace falta una base en archivo, así que la creamos en una carpeta temporal que se borra al terminar:

In [12]:
with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "prueba.db"

    with closing(sqlite3.connect(ruta)) as c:
        c.execute("CREATE TABLE Nota (Texto TEXT)")
        c.execute("INSERT INTO Nota VALUES ('sin commit')")
        # se cierra sin commit

    with closing(sqlite3.connect(ruta)) as c:
        print("Filas tras reabrir:", c.execute("SELECT COUNT(*) FROM Nota").fetchone()[0])

Filas tras reabrir: 0


La tabla sí existe, porque en Python 3.11 `sqlite3` ejecuta `CREATE TABLE` fuera de una transacción y se confirma de inmediato. El `INSERT`, en cambio, abrió una transacción que nunca se confirmó.

**Creer que `with conn:` cierra la conexión.** Solo confirma o revierte. Para cerrar al salir del bloque se usa `with closing(sqlite3.connect(...)) as conn:`, como en la celda anterior.

## 10. Ejercicio

Una biblioteca quiere registrar su catálogo:

```python
libros = [
    ("978-0134685991", "Effective Java", "Bloch", 2018, 45.0),
    ("978-1491957660", "Python for Data Analysis", "McKinney", 2017, 39.5),
    ("978-1449355739", "Learning SQL", "Beaulieu", 2009, 29.9),
    ("978-0596009205", "Head First Java", "Sierra", 2005, 25.0),
    ("978-1098104030", "Python Data Science Handbook", "VanderPlas", 2022, 52.0),
]
```

1. Crea la tabla `Libro(ISBN, Titulo, Autor, Anio, Precio)` con `ISBN` como clave primaria y carga los datos con `executemany` dentro de un bloque `with`.
2. Escribe una función `libros_entre(conn, desde, hasta)` que devuelva un DataFrame con los libros publicados en ese rango de años, usando parámetros.
3. Aplica un 10 % de descuento a los libros anteriores a 2010 y confirma el cambio.
4. Respalda la base con `iterdump`, restáurala en otra conexión en memoria y comprueba que el precio total coincide en ambas.

### Solución

In [13]:
libros = [
    ("978-0134685991", "Effective Java", "Bloch", 2018, 45.0),
    ("978-1491957660", "Python for Data Analysis", "McKinney", 2017, 39.5),
    ("978-1449355739", "Learning SQL", "Beaulieu", 2009, 29.9),
    ("978-0596009205", "Head First Java", "Sierra", 2005, 25.0),
    ("978-1098104030", "Python Data Science Handbook", "VanderPlas", 2022, 52.0),
]

biblio = sqlite3.connect(":memory:")
biblio.execute("""
CREATE TABLE Libro (
    ISBN TEXT PRIMARY KEY, Titulo TEXT NOT NULL, Autor TEXT, Anio INTEGER, Precio REAL
)""")
with biblio:
    biblio.executemany("INSERT INTO Libro VALUES (?, ?, ?, ?, ?)", libros)


def libros_entre(conn, desde, hasta):
    """Libros publicados entre los años desde y hasta, ambos incluidos."""
    return pd.read_sql_query(
        "SELECT Titulo, Anio, Precio FROM Libro WHERE Anio BETWEEN ? AND ? ORDER BY Anio",
        conn, params=(desde, hasta))


libros_entre(biblio, 2005, 2017)

,Titulo,Anio,Precio
0,Head First Java,2005,25.0
1,Learning SQL,2009,29.9
2,Python for Data Analysis,2017,39.5


In [14]:
with biblio:
    cur = biblio.execute("UPDATE Libro SET Precio = ROUND(Precio * 0.9, 2) WHERE Anio < ?", (2010,))
print("Libros con descuento:", cur.rowcount)

texto = "\n".join(biblio.iterdump())
restaurada = sqlite3.connect(":memory:")
restaurada.executescript(texto)

total = "SELECT ROUND(SUM(Precio), 2) FROM Libro"
print("Total original:  ", biblio.execute(total).fetchone()[0])
print("Total restaurado:", restaurada.execute(total).fetchone()[0])

restaurada.close()
biblio.close()
conn.close()

Libros con descuento: 2
Total original:   185.91
Total restaurado: 185.91


## Resumen

- SQLite guarda la base en un archivo (o en memoria) sin servidor, y `sqlite3` viene incluido en Python.
- El flujo es `connect` → `cursor`/`execute` → `fetch...` → `commit` → `close`; `with conn:` confirma o revierte, pero no cierra.
- Los valores externos se pasan con parámetros (`?` o `:nombre`), nunca con f-strings: así se evita la inyección SQL y los errores por comillas.
- `executemany` inserta muchas filas de una vez, y combinado con `csv.DictReader` carga un CSV en pocas líneas.
- `iterdump` produce un respaldo en SQL que se restaura con `executescript`.

En el **Módulo 41** se importan archivos CSV con pandas (`to_sql`) y se responden preguntas con consultas de agregación.